# Customer Churn – Responsible Baseline Model

**Task 02: ML Problem Framing & Responsible Data Card**

This notebook is a reproducible starter. Place the official portal dataset in the same folder as `customer_churn.csv`. Inspect and set the target column below. Do not use the results for live decisions.

## 1. Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

DATA_PATH = "customer_churn.csv"
TARGET = "Churn"  # CHANGE this to the exact target column in your official CSV.
DROP_COLUMNS = []  # Add identifiers and post-outcome/leakage columns after review.
RANDOM_STATE = 42


## 2. Load and inspect the official dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(20))


## 3. Validate target and remove identifiers/leakage

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' not found. Available columns: {list(df.columns)}")

data = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns]).copy()
data = data.dropna(subset=[TARGET])

print("Target distribution:")
display(data[TARGET].value_counts(dropna=False))
print("Duplicate rows:", data.duplicated().sum())

# Confirm which label represents churn. If needed, map labels explicitly, e.g.
# data[TARGET] = data[TARGET].map({"No": 0, "Yes": 1})
# Do not proceed until the positive class is confirmed.
y = data[TARGET]
X = data.drop(columns=[TARGET])

# Remove obvious unique identifiers only after checking column meaning.
high_cardinality = [c for c in X.columns if X[c].nunique(dropna=True) == len(X)]
print("Potential unique-ID columns to review:", high_cardinality)


## 4. Train/test split and preprocessing pipeline

Split before fitting imputers, scalers, or encoders to prevent preprocessing leakage. For time-dependent data, replace the random split with a time-based split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])


## 5. Non-ML baseline and simple ML baseline

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Dummy baseline")
print(classification_report(y_test, dummy_pred, zero_division=0))
print("Logistic regression baseline")
print(classification_report(y_test, pred, zero_division=0))


## 6. Evaluation and error review

In [ ]:
print("Confusion matrix [true rows, predicted columns]:")
print(confusion_matrix(y_test, pred))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Customer churn – baseline confusion matrix")
plt.show()

print("Precision:", precision_score(y_test, pred, average="binary", zero_division=0))
print("Recall:", recall_score(y_test, pred, average="binary", zero_division=0))
print("F1:", f1_score(y_test, pred, average="binary", zero_division=0))

# ROC-AUC is meaningful only when there are exactly two classes.
if y_test.nunique() == 2 and hasattr(model, "predict_proba"):
    positive_scores = model.predict_proba(X_test)[:, 1]
    print("ROC-AUC:", roc_auc_score(y_test, positive_scores))
else:
    print("ROC-AUC skipped: test target is not binary.")


## 7. Responsible-use checklist

- Confirmed data source, owner, purpose, permissions and retention: **record evidence**.
- Confirmed target definition, positive class, observation and prediction windows.
- Removed direct identifiers and reviewed proxy/sensitive attributes.
- Checked for post-outcome features and other leakage.
- Evaluated missingness, duplicates, class balance and representativeness.
- Compared against a transparent non-ML baseline.
- Documented false-positive and false-negative costs with the business owner.
- Human review is required; model output is not an automatic decision.
- Defined abstention, monitoring, incident response and rollback thresholds.

**Important:** This notebook is an educational baseline only. Do not deploy or contact customers based on its output without formal review and approval.